# Run an experiment

Pick a **model**, the **data**, and the **training recipe**, then train. Every run is saved automatically
(`runs/<site>/<run_name>/<timestamp>/`: trained model, config, metrics, predictions) and adds one row to
`results/experiment_log.csv`. Edit only the cells marked **CHOOSE** and run top to bottom.

How each model was trained in its paper: `docs/training_recipes.md` (PDF in `Resources/`). Section 1 can load that
recipe as the starting point.

| Step | What you choose | Code |
|---|---|---|
| 1 Model | architecture, hyperparameters, optionally the paper's recipe | `pvf/models/architectures/`, `pvf/training/paper_recipes.py` |
| 2 Data | site, history / future features, lookback, horizon, split | `pvf/data/` |
| 3 Input adapter | (inspect) how the batch becomes this model's input | `pvf/model_inputs/` |
| 4 Recipe | loss, optimizer, scheduler, epochs, early stopping, transfer... | `pvf/training/` |
| 5-8 | train, diagnose, compare, search | `pvf/experiment.py`, `pvf/evaluation/`, `pvf/search.py` |

In [ ]:
%matplotlib inline
import pandas as pd
import pvf

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

## 0 Machine check (once per machine)

In [ ]:
pvf.check_environment()

## 1 Model
Every model with its hyperparameters, and how each paper trained it.

In [ ]:
pvf.describe_models()

In [ ]:
pvf.describe_paper_recipes()

In [ ]:
# CHOOSE the model
model_name = "hdcf_mamba"
use_paper_recipe = True                 # start from the paper's hyperparameters + training recipe + lookback

preset = pvf.paper_recipe(model_name) if use_paper_recipe else None
model_hyperparameters = dict(preset.model_hyperparameters) if preset else {}
model_hyperparameters.update({})        # your changes, e.g. {"model_size": 64}
residual_on_smart_persistence = False   # True: forecast = smart persistence + learned correction
if preset:
    print("paper:", preset.source, "| hyperparameters:", model_hyperparameters)
    for note in preset.notes:
        print("  note:", note)

## 2 Data
Available features: `usable_as_future_feature` = known in advance. `*_oracle` = measured values used as a perfect forecast (an upper bound only).

In [ ]:
pvf.describe_features()

In [ ]:
# CHOOSE the data
data_config = pvf.DataConfig(
    site="ssp",                                   # "ssp" (Sirajganj, 2025) | "nist" (Gaithersburg, 2017)
    history_feature_names=["ac_power", "irradiance", "module_temperature", "clear_sky_irradiance",
                           "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    future_feature_names=["clear_sky_irradiance", "ac_power_yesterday",
                          "hour_of_day_sin", "hour_of_day_cos", "day_of_year_sin", "day_of_year_cos"],
    lookback_steps=preset.lookback_steps("30min") if preset else 128,   # x 30 min of history
    horizon_steps=48,                             # x 30 min ahead (48 = 24 h)
    split=pvf.SplitConfig(mode="blocked"),        # weekly blocks: 4 train / 1 validation / 1 test, every season
)
data_bundle = pvf.build_data_bundle(data_config)
pvf.describe_data_bundle(data_bundle)

In [ ]:
model = pvf.build_model(model_name, data_bundle.spec, residual_on_smart_persistence, **model_hyperparameters)
model

## 3 Input adapter
How the standard batch from the data pipeline becomes this model's input, step by step.

In [ ]:
batch = pvf.sample_batch(data_bundle)
print(model.input_adapter)
model.input_adapter.trace(batch)

In [ ]:
model.input_adapter.plot(batch);

## 4 Training recipe

In [ ]:
print("losses:", list(pvf.LOSSES), "| optimizers:", list(pvf.OPTIMIZERS), "| schedulers:", list(pvf.SCHEDULERS))
recipe = preset.recipe if preset else pvf.TrainingRecipe()

# CHOOSE: change any field (all fields: pvf.TrainingRecipe?)
recipe.n_epochs = 100
# recipe.loss = "huber";              recipe.loss_options = {"delta": 0.05}
# recipe.optimizer = "adamw";         recipe.learning_rate = 1e-3;  recipe.weight_decay = 1e-4
# recipe.scheduler = "one_cycle";     recipe.scheduler_options = {}
# recipe.warmup_epochs = 3
# recipe.early_stopping_metric = "nrmse";  recipe.early_stopping_patience = 15   # None = train all epochs
# recipe.batch_size = 32;             recipe.gradient_accumulation_steps = 2
# recipe.train_fraction = 0.2                                                    # data-efficiency curve
# recipe.initialize_from_run = r"runs/nist/hdcf_mamba_nist/<timestamp>"          # transfer from another run
# recipe.freeze_parameter_prefixes = ["encoder."]
run_name = f"{model_name}_{data_config.site}"      # label used in comparisons
recipe

## 5 Train and score
Saves the trained model and all results; prints standard metrics per split.

In [ ]:
experiment_config = pvf.ExperimentConfig(model_name=model_name, model_hyperparameters=model_hyperparameters,
                                         residual_on_smart_persistence=residual_on_smart_persistence,
                                         data=data_config, recipe=recipe, run_name=run_name)
result = pvf.run_experiment(experiment_config, data_bundle=data_bundle, model=model)
result.summary()

## 6 Diagnose this run

In [ ]:
pvf.plot_training_history(result.run_dir);

In [ ]:
pvf.plot_forecast_examples(result.run_dir, split="test", n_examples=3);

In [ ]:
pvf.plot_per_horizon([result.run_dir], metric="nrmse_pct", split="test");

In [ ]:
pvf.plot_error_by_hour([result.run_dir], split="test");

## 7 Compare runs
Latest run per run name, best first. Column meanings: `pvf.evaluation.metrics.METRIC_DESCRIPTIONS`.

In [ ]:
comparison = pvf.compare_runs(site=data_config.site, split="test")
comparison

In [ ]:
best_runs = comparison.run_dir[:5]
pvf.plot_per_horizon(best_runs, metric="nrmse_pct", split="test");
pvf.plot_error_by_hour(best_runs, split="test");

## 8 Hyperparameter search (optional)
The papers tuned with Optuna, grid search or Nixtla. Keys are `model.<hyperparameter>` or `recipe.<field>`; every
trial is a saved run, ranked on the **validation** split.

In [ ]:
# CHOOSE the search space, then set run_search = True
run_search = False
if run_search:
    search_results = pvf.search_hyperparameters(experiment_config, {
        "model.model_size": [64, 128],
        "recipe.learning_rate": [1e-4, 3e-4, 1e-3],
    }, n_trials=6, data_bundle=data_bundle)
    display(search_results)

## 9 Fusing models
`pvf/models/architectures/fusion_example.py` fuses iTransformer's variable encoder with HDCF-Mamba's temporal encoder
through a learned gate (4-step recipe in its docstring). Try it with `model_name = "fusion_itransformer_hdcf"`.
To build your own: copy that file, swap in two other encoders, register it under a new name.

## 10 Reload a saved model

In [ ]:
trained_model = pvf.load_trained_model(result.run_dir)        # ready to call on any ForecastBatch